In [1]:
from pathlib import Path
import json
import math
import re
import unicodedata

import pandas as pd
from IPython.display import display

BENCHMARK_VERSION = 'v1'
MODEL_NAME = 'BAAI/bge-m3'
KS = (1, 3, 5, 10)
EXPECTED_CASES = 50
EXPECTED_TOP_K = 10

def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in candidates:
        if (root / 'bge' / 'retrieval_top10_bge_m3.json').exists() and (root / 'data').exists():
            return root.resolve()
    raise FileNotFoundError(
        'Không tìm thấy thư mục dự án. Hãy chạy notebook trong project hoặc thư mục embedding.'
    )

PROJECT_ROOT = find_project_root()
RETRIEVAL_PATH = PROJECT_ROOT / 'bge' / 'retrieval_top10_bge_m3.json'
GOLD_PATH = PROJECT_ROOT / 'data' / 'ALQAC2026_public_test.json'
CORPUS_PATH = PROJECT_ROOT / 'data' / 'corpus_law_pub.json'
PREDICTION_PATH = PROJECT_ROOT / 'bge' / 'predictions_llama-3-2-3b_seed-2026.json'
OUTPUT_DIR = PROJECT_ROOT / 'embedding' / 'retrieval_eval_bge_m3'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT :', PROJECT_ROOT)
print('RETRIEVAL    :', RETRIEVAL_PATH)
print('GOLD         :', GOLD_PATH)
print('CORPUS       :', CORPUS_PATH)
print('OUTPUT_DIR   :', OUTPUT_DIR)

PROJECT_ROOT : C:\Users\DELL\Desktop\4ngaydemnguocdenbinhminh
RETRIEVAL    : C:\Users\DELL\Desktop\4ngaydemnguocdenbinhminh\bge\retrieval_top10_bge_m3.json
GOLD         : C:\Users\DELL\Desktop\4ngaydemnguocdenbinhminh\data\ALQAC2026_public_test.json
CORPUS       : C:\Users\DELL\Desktop\4ngaydemnguocdenbinhminh\data\corpus_law_pub.json
OUTPUT_DIR   : C:\Users\DELL\Desktop\4ngaydemnguocdenbinhminh\embedding\retrieval_eval_bge_m3


In [2]:
def load_json(path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

retrieval = load_json(RETRIEVAL_PATH)
gold_rows = load_json(GOLD_PATH)
corpus = load_json(CORPUS_PATH)

print(f'Retrieval cases: {len(retrieval)}')
print(f'Gold cases     : {len(gold_rows)}')
print(f'Corpus laws    : {len(corpus)}')
assert len(gold_rows) == EXPECTED_CASES, f'Gold phải có {EXPECTED_CASES} case'

Retrieval cases: 50
Gold cases     : 50
Corpus laws    : 18


In [3]:
def normalize_text(value):
    value = unicodedata.normalize('NFD', str(value or ''))
    value = ''.join(ch for ch in value if unicodedata.category(ch) != 'Mn')
    value = value.replace('đ', 'd').replace('Đ', 'D').lower()
    return re.sub(r'\s+', ' ', value).strip()

corpus_by_key = {}
for law in corpus:
    for article_no, article in enumerate(law['content'], start=1):
        corpus_by_key[(str(law['law_id']), article_no)] = {
            'aid': int(article['aid']),
            'content': str(article.get('content_Article', '')),
        }

CANONICAL_TO_CORPUS_LAW_ID = {
    'BLTTDS_2015': '92/2015/QH13',
    'BLDS_2015': '91/2015/QH13',
    'HNGD_2014': '52/2014/QH13',
    'DAT_DAI_2013': '45/2013/QH13',
    'THADS_2008': '26/2008/QH12',
    'HO_TICH_2014': '60/2014/QH13',
    'TCTD_2010': '47/2010/QH12',
    'XAY_DUNG_2014': '50/2014/QH13',
    'TTHC_2015': '93/2015/QH13',
    'KHIEU_NAI_2011': '02/2011/QH13',
    'KDBDS_2014': '66/2014/QH13',
    'NQ326_2016': '326/2016/UBTVQH14',
    'ND37_2015': '37/2015/NĐ-CP',
}

def first_explicit_year(text, *years):
    return next((year for year in years if str(year) in text), None)

def canonical_gold_law(title):
    value = normalize_text(title)
    if 'bo luat to tung dan su' in value or value == 'bo luat to tung':
        return f"BLTTDS_{first_explicit_year(value, 2015) or 2015}"
    if 'bo luat dan su' in value:
        return f"BLDS_{first_explicit_year(value, 1995, 2005, 2015) or 2015}"
    if 'luat hon nhan va gia dinh' in value or 'luat hon nhan gia dinh' in value:
        return f"HNGD_{first_explicit_year(value, 2000, 2014) or 2014}"
    if 'luat dat dai' in value:
        return f"DAT_DAI_{first_explicit_year(value, 1987, 1993, 2003, 2013) or 2013}"
    if 'luat thi hanh an dan su' in value:
        return 'THADS_2008'
    if 'luat ho tich' in value:
        return 'HO_TICH_2014'
    if 'luat cac to chuc tin dung' in value:
        return 'TCTD_2010'
    if 'luat xay dung' in value:
        return f"XAY_DUNG_{first_explicit_year(value, 2003, 2014) or 2014}"
    if 'luat to tung hanh chinh' in value:
        return 'TTHC_2015'
    if 'luat khieu nai, to cao' in value or 'luat khieu nai to cao' in value:
        return f"KHIEU_NAI_TO_CAO_{first_explicit_year(value, 1998, 2006) or 'OLD'}"
    if 'luat khieu nai' in value:
        return 'KHIEU_NAI_2011'
    if 'luat kinh doanh bat dong san' in value:
        return f"KDBDS_{first_explicit_year(value, 2006, 2014) or 2014}"
    if '326/2016' in value or re.search(r'nghi quyet (so )?:? ?326\b', value):
        return 'NQ326_2016'
    if '37/2015/nd-cp' in value:
        return 'ND37_2015'
    if '181/2004/nd-cp' in value:
        return 'ND181_2004'
    if 'phap lenh' in value and 'an phi' in value:
        return 'PHAP_LENH_AN_PHI_2009'
    return re.sub(r'ngay .*$', '', value).rstrip(';, .')

def parse_gold(row):
    all_gold = set()
    available_gold = set()
    unavailable_gold = set()
    for raw_line in str(row.get('related_law_provisions', '')).splitlines():
        line = raw_line.strip()
        if not line:
            continue
        article_numbers = [int(x) for x in re.findall(r'\bdieu\s+(\d+)', normalize_text(line))]
        if not article_numbers:
            continue
        title = line.split('|', 1)[0].strip()
        canonical = canonical_gold_law(title)
        corpus_law_id = CANONICAL_TO_CORPUS_LAW_ID.get(canonical)
        for article_no in article_numbers:
            canonical_key = (canonical, article_no)
            all_gold.add(canonical_key)
            exact_key = (corpus_law_id, article_no) if corpus_law_id else None
            if exact_key and exact_key in corpus_by_key:
                available_gold.add(exact_key)
            else:
                unavailable_gold.add(canonical_key)
    return {
        'all': all_gold,
        'available': available_gold,
        'unavailable': unavailable_gold,
    }

In [4]:
gold_ids = [row['case_id'] for row in gold_rows]
retrieval_ids = list(retrieval)
issues = {
    'missing_case_ids': [case_id for case_id in gold_ids if case_id not in retrieval],
    'extra_case_ids': [case_id for case_id in retrieval_ids if case_id not in set(gold_ids)],
    'wrong_topk_length': [],
    'duplicate_keys_within_case': [],
    'bad_ranks': [],
    'non_descending_scores': [],
    'corpus_key_mismatches': [],
    'aid_mismatches': [],
    'content_mismatches': [],
}

for case_id, items in retrieval.items():
    if not isinstance(items, list) or len(items) != EXPECTED_TOP_K:
        issues['wrong_topk_length'].append(case_id)
        continue
    keys = [(str(item['law_id']), int(item['article_no'])) for item in items]
    if len(set(keys)) != len(keys):
        issues['duplicate_keys_within_case'].append(case_id)
    if any(int(item['rank']) != index for index, item in enumerate(items, start=1)):
        issues['bad_ranks'].append(case_id)
    scores = [float(item['score']) for item in items]
    if any(current > previous for previous, current in zip(scores, scores[1:])):
        issues['non_descending_scores'].append(case_id)
    for item, key in zip(items, keys):
        article = corpus_by_key.get(key)
        label = f'{case_id}:{key[0]}|{key[1]}'
        if article is None:
            issues['corpus_key_mismatches'].append(label)
            continue
        if int(item['aid']) != article['aid']:
            issues['aid_mismatches'].append(label)
        if str(item.get('content_Article', '')) != article['content']:
            issues['content_mismatches'].append(label)

integrity_df = pd.DataFrame([
    {'check': name, 'n_errors': len(values), 'examples': values[:5]}
    for name, values in issues.items()
])
display(integrity_df)

assert len(retrieval) == EXPECTED_CASES, f'Retrieval phải có {EXPECTED_CASES} case'
assert not any(issues.values()), 'File retrieval chưa hợp lệ; xem bảng integrity_df ở trên.'
print(f'PASS: file hợp lệ, đủ {EXPECTED_CASES} case × {EXPECTED_TOP_K} kết quả.')

,check,n_errors,examples
0,missing_case_ids,0,[]
1,extra_case_ids,0,[]
2,wrong_topk_length,0,[]
3,duplicate_keys_within_case,0,[]
4,bad_ranks,0,[]
5,non_descending_scores,0,[]
6,corpus_key_mismatches,0,[]
7,aid_mismatches,0,[]
8,content_mismatches,0,[]


PASS: file hợp lệ, đủ 50 case × 10 kết quả.


In [5]:
def safe_mean(values):
    values = [value for value in values if value is not None and math.isfinite(value)]
    return sum(values) / len(values) if values else None

def evaluate_case(gold_row):
    case_id = gold_row['case_id']
    gold = parse_gold(gold_row)
    ranked_keys = [
        (str(item['law_id']), int(item['article_no']))
        for item in retrieval[case_id]
    ]
    result = {
        'case_id': case_id,
        'gold_all': len(gold['all']),
        'gold_available': len(gold['available']),
        'gold_unavailable': len(gold['unavailable']),
    }
    for k in KS:
        relevance = [key in gold['available'] for key in ranked_keys[:k]]
        hits = sum(relevance)
        first_hit = next((index for index, flag in enumerate(relevance, start=1) if flag), None)

        dcg = sum(1 / math.log2(index + 1) for index, flag in enumerate(relevance, start=1) if flag)
        idcg = sum(1 / math.log2(index + 1) for index in range(1, min(len(gold['available']), k) + 1))

        running_hits = 0
        precision_sum = 0.0
        for index, flag in enumerate(relevance, start=1):
            if flag:
                running_hits += 1
                precision_sum += running_hits / index

        result[f'hits@{k}'] = hits
        result[f'precision@{k}'] = hits / k
        result[f'recall_available@{k}'] = hits / len(gold['available']) if gold['available'] else None
        result[f'recall_end_to_end@{k}'] = hits / len(gold['all']) if gold['all'] else None
        result[f'hit@{k}'] = int(hits > 0)
        result[f'rr@{k}'] = 1 / first_hit if first_hit else 0.0
        result[f'ndcg@{k}'] = dcg / idcg if idcg else None
        result[f'ap@{k}'] = precision_sum / len(gold['available']) if gold['available'] else None
    return result

per_case_df = pd.DataFrame(evaluate_case(row) for row in gold_rows)
display(per_case_df.head())

,case_id,gold_all,gold_available,gold_unavailable,hits@1,precision@1,recall_available@1,recall_end_to_end@1,hit@1,rr@1,...,ndcg@5,ap@5,hits@10,precision@10,recall_available@10,recall_end_to_end@10,hit@10,rr@10,ndcg@10,ap@10
0,case_4101,6,6,0,1,1.0,0.166667,0.166667,1,1.0,...,0.470365,0.233333,2,0.2,0.333333,0.333333,1,1.0,0.419665,0.233333
1,case_4337,7,1,6,0,0.0,0.000000,0.000000,0,0.0,...,0.000000,0.000000,0,0.0,0.000000,0.000000,0,0.0,0.000000,0.000000
2,case_4588,14,10,4,0,0.0,0.000000,0.000000,0,0.0,...,0.000000,0.000000,0,0.0,0.000000,0.000000,0,0.0,0.000000,0.000000
3,case_4616,19,12,7,0,0.0,0.000000,0.000000,0,0.0,...,0.000000,0.000000,0,0.0,0.000000,0.000000,0,0.0,0.000000,0.000000
4,case_6616,28,27,1,0,0.0,0.000000,0.000000,0,0.0,...,0.131205,0.007407,1,0.1,0.037037,0.035714,1,0.2,0.085143,0.007407


In [6]:
total_gold = int(per_case_df['gold_all'].sum())
total_available = int(per_case_df['gold_available'].sum())
corpus_coverage = total_available / total_gold

rows = []
for k in KS:
    hits = int(per_case_df[f'hits@{k}'].sum())
    rows.append({
        'K': k,
        'Hits': hits,
        'Precision': hits / (len(per_case_df) * k),
        'Recall macro (corpus-conditioned)': per_case_df[f'recall_available@{k}'].mean(),
        'Recall micro (corpus-conditioned)': hits / total_available,
        'Recall macro (end-to-end)': per_case_df[f'recall_end_to_end@{k}'].mean(),
        'Recall micro (end-to-end)': hits / total_gold,
        'Hit Rate': per_case_df[f'hit@{k}'].mean(),
        'MRR': per_case_df[f'rr@{k}'].mean(),
        'nDCG': per_case_df[f'ndcg@{k}'].mean(),
        'MAP': per_case_df[f'ap@{k}'].mean(),
    })

metrics_df = pd.DataFrame(rows).set_index('K')
display(metrics_df.style.format('{:.2%}', subset=[column for column in metrics_df if column != 'Hits']))

print(f'Tổng gold article citation     : {total_gold}')
print(f'Gold tồn tại trong corpus      : {total_available}')
print(f'Gold không tồn tại trong corpus: {total_gold - total_available}')
print(f'Corpus coverage                : {corpus_coverage:.2%}')

,Hits,Precision,Recall macro (corpus-conditioned),Recall micro (corpus-conditioned),Recall macro (end-to-end),Recall micro (end-to-end),Hit Rate,MRR,nDCG,MAP
K,,,,,,,,,,
1,5,10.00%,0.89%,0.95%,0.89%,0.74%,10.00%,10.00%,10.00%,0.89%
3,9,6.00%,1.39%,1.70%,1.39%,1.33%,14.00%,12.00%,6.82%,1.18%
5,13,5.20%,2.19%,2.46%,2.02%,1.93%,20.00%,13.40%,6.03%,1.43%
10,26,5.20%,4.05%,4.91%,3.84%,3.85%,38.00%,15.76%,6.04%,1.74%


Tổng gold article citation     : 675
Gold tồn tại trong corpus      : 529
Gold không tồn tại trong corpus: 146
Corpus coverage                : 78.37%


In [7]:
main_scores = pd.DataFrame({
    'Metric': ['Precision@10', 'Recall@10 macro (corpus-conditioned)', 'Hit Rate@10', 'MRR@10', 'nDCG@10', 'MAP@10'],
    'Score': [
        metrics_df.loc[10, 'Precision'],
        metrics_df.loc[10, 'Recall macro (corpus-conditioned)'],
        metrics_df.loc[10, 'Hit Rate'],
        metrics_df.loc[10, 'MRR'],
        metrics_df.loc[10, 'nDCG'],
        metrics_df.loc[10, 'MAP'],
    ],
})
display(main_scores.style.format({'Score': '{:.2%}'}))

,Metric,Score
0,Precision@10,5.20%
1,Recall@10 macro (corpus-conditioned),4.05%
2,Hit Rate@10,38.00%
3,MRR@10,15.76%
4,nDCG@10,6.04%
5,MAP@10,1.74%


In [8]:
hit_cases_df = per_case_df[per_case_df['hit@10'] == 1].copy()
no_hit_cases_df = per_case_df[per_case_df['hit@10'] == 0].copy()
best_cases_df = per_case_df.sort_values(['hits@10', 'rr@10'], ascending=False).head(10)

print(f'Case có ít nhất 1 hit trong top 10: {len(hit_cases_df)}/{len(per_case_df)}')
print(f'Case không có hit trong top 10 : {len(no_hit_cases_df)}/{len(per_case_df)}')
display(best_cases_df[['case_id', 'gold_all', 'gold_available', 'hits@10', 'precision@10', 'recall_available@10', 'rr@10']])
display(no_hit_cases_df[['case_id', 'gold_all', 'gold_available', 'gold_unavailable']])

Case có ít nhất 1 hit trong top 10: 19/50
Case không có hit trong top 10 : 31/50


,case_id,gold_all,gold_available,hits@10,precision@10,recall_available@10,rr@10
29,case_8147,17,17,3,0.3,0.176471,0.500
0,case_4101,6,6,2,0.2,0.333333,1.000
5,case_5226,15,15,2,0.2,0.133333,1.000
16,case_8758,22,22,2,0.2,0.090909,1.000
43,case_4557,18,18,2,0.2,0.111111,0.250
6,case_6284,14,14,2,0.2,0.142857,0.125
14,case_5045,13,13,1,0.1,0.076923,1.000
48,case_4644,11,11,1,0.1,0.090909,1.000
36,case_7467,16,16,1,0.1,0.062500,0.500
15,case_4920,17,7,1,0.1,0.142857,0.250


,case_id,gold_all,gold_available,gold_unavailable
1,case_4337,7,1,6
2,case_4588,14,10,4
3,case_4616,19,12,7
7,case_3995,6,6,0
9,case_8219,15,12,3
10,case_2705,8,7,1
13,case_5658,15,15,0
18,case_127,33,10,23
19,case_2978,4,4,0
20,case_4579,7,3,4


In [9]:
summary = {
    'benchmark_version': BENCHMARK_VERSION,
    'retriever': MODEL_NAME,
    'matching_policy': 'exact law version + article number',
    'n_cases': len(per_case_df),
    'top_k': EXPECTED_TOP_K,
    'total_gold_article_citations': total_gold,
    'gold_article_citations_available_in_corpus': total_available,
    'corpus_coverage': corpus_coverage,
    'metrics': {str(k): {key: (int(value) if key == 'Hits' else float(value))
                         for key, value in metrics_df.loc[k].items()}
                for k in KS},
}

metrics_path = OUTPUT_DIR / 'retrieval_metrics_bge_m3.json'
per_case_path = OUTPUT_DIR / 'retrieval_per_case_bge_m3.csv'
no_hit_path = OUTPUT_DIR / 'retrieval_no_hit_cases_bge_m3.csv'
integrity_path = OUTPUT_DIR / 'retrieval_integrity_bge_m3.csv'

with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
per_case_df.to_csv(per_case_path, index=False, encoding='utf-8-sig')
no_hit_cases_df.to_csv(no_hit_path, index=False, encoding='utf-8-sig')
integrity_df.to_csv(integrity_path, index=False, encoding='utf-8-sig')

print('Đã lưu:')
for path in (metrics_path, per_case_path, no_hit_path, integrity_path):
    print(' -', path)

Đã lưu:
 - C:\Users\DELL\Desktop\4ngaydemnguocdenbinhminh\embedding\retrieval_eval_bge_m3\retrieval_metrics_bge_m3.json
 - C:\Users\DELL\Desktop\4ngaydemnguocdenbinhminh\embedding\retrieval_eval_bge_m3\retrieval_per_case_bge_m3.csv
 - C:\Users\DELL\Desktop\4ngaydemnguocdenbinhminh\embedding\retrieval_eval_bge_m3\retrieval_no_hit_cases_bge_m3.csv
 - C:\Users\DELL\Desktop\4ngaydemnguocdenbinhminh\embedding\retrieval_eval_bge_m3\retrieval_integrity_bge_m3.csv


In [10]:
if PREDICTION_PATH.exists():
    predictions = load_json(PREDICTION_PATH)
    different_cases = []
    for case_id, source_items in retrieval.items():
        prediction_items = predictions.get(case_id, {}).get('retrieved_laws', [])
        source_signature = [
            (item['rank'], item['law_id'], item['aid'], item['article_no'])
            for item in source_items
        ]
        prediction_signature = [
            (item['rank'], item['law_id'], item['aid'], item['article_no'])
            for item in prediction_items
        ]
        if source_signature != prediction_signature:
            different_cases.append(case_id)
    print(f'Case khác retrieval trong output Llama: {len(different_cases)}/{len(retrieval)}')
    if different_cases:
        print(different_cases)
else:
    print('Không có prediction file; bỏ qua kiểm tra tùy chọn này.')

Case khác retrieval trong output Llama: 0/50
